# 2️⃣ Data Preprocessing & Feature Engineering

## Clean, transform, and prepare data for modeling

### Load Raw Data

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

# Load raw data
df = pd.read_csv("../data/raw/Energy_consumption.csv")
print(f"Raw data shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Raw data shape: (1000, 11)
Columns: ['Timestamp', 'Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'HVACUsage', 'LightingUsage', 'RenewableEnergy', 'DayOfWeek', 'Holiday', 'EnergyConsumption']


### Feature Engineering: Extract Temporal Features

In [2]:
# Convert Timestamp to datetime
df["Timestamp"] = pd.to_datetime(df["Timestamp"])

# Extract temporal features
df["hour"] = df["Timestamp"].dt.hour
df["day"] = df["Timestamp"].dt.day
df["month"] = df["Timestamp"].dt.month
df["year"] = df["Timestamp"].dt.year

# Drop original Timestamp (we extracted what we need)
df = df.drop("Timestamp", axis=1)

print("✓ Temporal features extracted")
print(f"New columns: {list(df.columns)}")

✓ Temporal features extracted
New columns: ['Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'HVACUsage', 'LightingUsage', 'RenewableEnergy', 'DayOfWeek', 'Holiday', 'EnergyConsumption', 'hour', 'day', 'month', 'year']


### Feature Engineering: Create Derived Features

In [3]:
# Create HeatIndex (Temperature + Humidity interaction)
df["HeatIndex"] = df["Temperature"] + 0.5 * df["Humidity"]

print("✓ Derived features created")
print(f"HeatIndex range: {df['HeatIndex'].min():.2f} - {df['HeatIndex'].max():.2f}")

✓ Derived features created
HeatIndex range: 35.24 - 59.47


### Define Feature Lists

In [4]:
# Define feature groups
categorical_features = ["HVACUsage", "LightingUsage", "DayOfWeek", "Holiday"]
numeric_features = ["Temperature", "Humidity", "SquareFootage", "Occupancy", "RenewableEnergy", "HeatIndex"]
target = "EnergyConsumption"

print(f"\nCategorical Features ({len(categorical_features)}): {categorical_features}")
print(f"Numeric Features ({len(numeric_features)}): {numeric_features}")
print(f"Target Variable: {target}")


Categorical Features (4): ['HVACUsage', 'LightingUsage', 'DayOfWeek', 'Holiday']
Numeric Features (6): ['Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'RenewableEnergy', 'HeatIndex']
Target Variable: EnergyConsumption


### Create Preprocessing Pipelines

In [5]:
# Numeric pipeline: impute missing values + scale
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical pipeline: impute missing values + one-hot encode
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both pipelines
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("✓ Preprocessing pipelines created")
print(f"  - Numeric: Imputation (median) + Scaling")
print(f"  - Categorical: Imputation (constant) + One-Hot Encoding")

✓ Preprocessing pipelines created
  - Numeric: Imputation (median) + Scaling
  - Categorical: Imputation (constant) + One-Hot Encoding


### Prepare Features and Target

In [6]:
# Separate features and target
X = df[numeric_features + categorical_features]
y = df[target]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeatures: {list(X.columns)}")

Features shape: (1000, 10)
Target shape: (1000,)

Features: ['Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'RenewableEnergy', 'HeatIndex', 'HVACUsage', 'LightingUsage', 'DayOfWeek', 'Holiday']


### Train-Test Split

In [7]:
# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"✓ Data split complete")
print(f"\nTraining set:")
print(f"  X_train: {X_train.shape}")
print(f"  y_train: {y_train.shape}")
print(f"\nTest set:")
print(f"  X_test: {X_test.shape}")
print(f"  y_test: {y_test.shape}")

✓ Data split complete

Training set:
  X_train: (800, 10)
  y_train: (800,)

Test set:
  X_test: (200, 10)
  y_test: (200,)


### Data Quality Check

In [8]:
# Check for missing values in train set
print(f"Missing values in X_train:")
print(X_train.isnull().sum())

# Check data types
print(f"\nData types in X_train:")
print(X_train.dtypes)

print(f"\n✓ Preprocessing complete and ready for modeling")

Missing values in X_train:
Temperature        0
Humidity           0
SquareFootage      0
Occupancy          0
RenewableEnergy    0
HeatIndex          0
HVACUsage          0
LightingUsage      0
DayOfWeek          0
Holiday            0
dtype: int64

Data types in X_train:
Temperature        float64
Humidity           float64
SquareFootage      float64
Occupancy            int64
RenewableEnergy    float64
HeatIndex          float64
HVACUsage              str
LightingUsage          str
DayOfWeek              str
Holiday                str
dtype: object

✓ Preprocessing complete and ready for modeling


### Summary

✓ Extracted temporal features (hour, day, month, year)  
✓ Created derived feature (HeatIndex)  
✓ Built preprocessing pipelines for numeric & categorical features  
✓ Split data: 80% train, 20% test  
✓ Ready for model training